# Phase 8 — Vocal Emotion Training (ECAPA-TDNN)

**Model:** ECAPA-TDNN (SpeechBrain, 6.2M params) + MLP emotion head

**Datasets:** RAVDESS (~200MB) + CREMA-D (~1.5GB)

**5 classes:** Neutral, Enthusiastic, Nervous, Angry, Sad

**Two-phase training:**
- Phase 1: Freeze encoder, train head only (~15 min on T4)
- Phase 2: Unfreeze top TDNN layers, fine-tune end-to-end (~1-2 hrs on T4)

**Target:** UAR (Unweighted Average Recall) >= 0.58

**Setup:** Runtime -> Change runtime type -> **T4 GPU**

## Cell 1: Install Dependencies & GPU Check

In [ ]:
!pip install -q torch torchaudio transformers
!pip install -q speechbrain
!pip install -q soundfile librosa pandas scikit-learn

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU! Go to Runtime -> Change runtime type -> T4 GPU")

## Cell 2: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = "/content/drive/MyDrive/voice_pipeline_models/vocal_emotion"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Models will be saved to: {SAVE_DIR}")

## Cell 3: Download RAVDESS Dataset

RAVDESS: 24 actors, 8 emotions, ~1440 speech clips (~200MB from Zenodo).

In [ ]:
import os

RAVDESS_DIR = "/content/ravdess"

if os.path.exists(RAVDESS_DIR) and len(os.listdir(RAVDESS_DIR)) >= 24:
    print(f"RAVDESS already downloaded: {len(os.listdir(RAVDESS_DIR))} folders")
else:
    os.makedirs(RAVDESS_DIR, exist_ok=True)
    print("Downloading RAVDESS from Zenodo (~200MB)...")
    !wget -q --show-progress -O /content/ravdess.zip \
        "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"
    !unzip -q -o /content/ravdess.zip -d {RAVDESS_DIR}
    !rm /content/ravdess.zip

    # Check structure — might be nested
    contents = os.listdir(RAVDESS_DIR)
    if len(contents) == 1 and os.path.isdir(os.path.join(RAVDESS_DIR, contents[0])):
        # Nested directory — move contents up
        nested = os.path.join(RAVDESS_DIR, contents[0])
        for item in os.listdir(nested):
            os.rename(os.path.join(nested, item), os.path.join(RAVDESS_DIR, item))
        os.rmdir(nested)

actor_dirs = [d for d in os.listdir(RAVDESS_DIR) if d.startswith('Actor')]
print(f"RAVDESS: {len(actor_dirs)} actor folders")

# Count total files
total = sum(len(os.listdir(os.path.join(RAVDESS_DIR, d))) for d in actor_dirs)
print(f"Total clips: {total}")

# Show sample filename
sample = os.listdir(os.path.join(RAVDESS_DIR, actor_dirs[0]))[0]
print(f"Sample filename: {sample}")

## Cell 4: Download CREMA-D Dataset

CREMA-D: 91 actors, 6 emotions, ~7442 clips (~1.5GB from GitHub).

In [ ]:
CREMAD_DIR = "/content/cremad"

if os.path.exists(CREMAD_DIR) and len([f for f in os.listdir(CREMAD_DIR) if f.endswith('.wav')]) > 5000:
    wav_count = len([f for f in os.listdir(CREMAD_DIR) if f.endswith('.wav')])
    print(f"CREMA-D already downloaded: {wav_count} WAV files")
else:
    os.makedirs(CREMAD_DIR, exist_ok=True)
    print("Downloading CREMA-D from GitHub (~1.5GB)...")
    print("This takes 5-10 minutes.\n")

    # Clone just the AudioWAV directory using sparse checkout
    !git clone --depth 1 --filter=blob:none --sparse \
        https://github.com/CheyneyComputerScience/CREMA-D.git /content/cremad_repo
    %cd /content/cremad_repo
    !git sparse-checkout set AudioWAV
    %cd /content

    # Move WAV files to our directory
    !mv /content/cremad_repo/AudioWAV/*.wav {CREMAD_DIR}/
    !rm -rf /content/cremad_repo

wav_count = len([f for f in os.listdir(CREMAD_DIR) if f.endswith('.wav')])
print(f"CREMA-D: {wav_count} WAV files")

# Show sample filename
sample = sorted([f for f in os.listdir(CREMAD_DIR) if f.endswith('.wav')])[0]
print(f"Sample filename: {sample}")

## Cell 5: Parse Datasets & Map Emotions to 5 Classes

**RAVDESS** filename format: `03-01-XX-01-01-01-01.wav` where XX is emotion code.

**CREMA-D** filename format: `1001_DFA_XXX_XX.wav` where XXX is emotion code.

We map both to our 5 classes: Neutral, Enthusiastic, Nervous, Angry, Sad.

In [ ]:
import os
import numpy as np
from collections import Counter

CLASSES = ["Neutral", "Enthusiastic", "Nervous", "Angry", "Sad"]
NUM_CLASSES = len(CLASSES)

# ─── RAVDESS emotion mapping ───
# RAVDESS codes: 01=neutral, 02=calm, 03=happy, 04=sad,
#                05=angry, 06=fearful, 07=disgust, 08=surprised
RAVDESS_MAP = {
    '01': 0,  # neutral -> Neutral
    '02': 0,  # calm -> Neutral
    '03': 1,  # happy -> Enthusiastic
    '04': 4,  # sad -> Sad
    '05': 3,  # angry -> Angry
    '06': 2,  # fearful -> Nervous
    '07': 3,  # disgust -> Angry (closest)
    '08': 1,  # surprised -> Enthusiastic (closest)
}

# ─── CREMA-D emotion mapping ───
# CREMA-D codes: ANG, DIS, FEA, HAP, NEU, SAD
CREMAD_MAP = {
    'NEU': 0,  # neutral -> Neutral
    'HAP': 1,  # happy -> Enthusiastic
    'SAD': 4,  # sad -> Sad
    'ANG': 3,  # angry -> Angry
    'FEA': 2,  # fearful -> Nervous
    'DIS': 3,  # disgust -> Angry (closest)
}


def parse_ravdess(ravdess_dir):
    """Parse RAVDESS dataset. Returns list of (filepath, label) tuples."""
    samples = []
    for actor_dir in sorted(os.listdir(ravdess_dir)):
        actor_path = os.path.join(ravdess_dir, actor_dir)
        if not os.path.isdir(actor_path):
            continue
        for fname in os.listdir(actor_path):
            if not fname.endswith('.wav'):
                continue
            # Format: 03-01-XX-01-01-01-01.wav
            parts = fname.replace('.wav', '').split('-')
            if len(parts) < 3:
                continue
            emotion_code = parts[2]
            if emotion_code in RAVDESS_MAP:
                label = RAVDESS_MAP[emotion_code]
                # Actor ID for speaker-aware splitting
                actor_id = f"ravdess_{actor_dir}"
                samples.append({
                    'path': os.path.join(actor_path, fname),
                    'label': label,
                    'speaker': actor_id,
                })
    return samples


def parse_cremad(cremad_dir):
    """Parse CREMA-D dataset. Returns list of (filepath, label) tuples."""
    samples = []
    for fname in sorted(os.listdir(cremad_dir)):
        if not fname.endswith('.wav'):
            continue
        # Format: 1001_DFA_ANG_XX.wav
        parts = fname.replace('.wav', '').split('_')
        if len(parts) < 3:
            continue
        actor_id_str = parts[0]
        emotion_code = parts[2]
        if emotion_code in CREMAD_MAP:
            label = CREMAD_MAP[emotion_code]
            samples.append({
                'path': os.path.join(cremad_dir, fname),
                'label': label,
                'speaker': f"cremad_{actor_id_str}",
            })
    return samples


# Parse both datasets
ravdess_samples = parse_ravdess(RAVDESS_DIR)
cremad_samples = parse_cremad(CREMAD_DIR)
all_samples = ravdess_samples + cremad_samples

print(f"RAVDESS: {len(ravdess_samples)} samples")
print(f"CREMA-D: {len(cremad_samples)} samples")
print(f"Total:   {len(all_samples)} samples")

# Class distribution
label_counts = Counter(s['label'] for s in all_samples)
print(f"\nClass distribution:")
for i, name in enumerate(CLASSES):
    count = label_counts.get(i, 0)
    print(f"  {name:15s}: {count:>5} ({100*count/len(all_samples):.1f}%)")

# Speaker stats
speakers = set(s['speaker'] for s in all_samples)
print(f"\nUnique speakers: {len(speakers)}")

## Cell 6: Speaker-Aware Train/Val/Test Split

In [ ]:
import numpy as np
from collections import Counter

speakers = sorted(set(s['speaker'] for s in all_samples))
print(f"Total speakers: {len(speakers)}")

# 80/10/10 split by speaker
np.random.seed(42)
speaker_indices = np.random.permutation(len(speakers))
n_train = int(0.8 * len(speakers))
n_val = int(0.1 * len(speakers))

train_speakers = set(speakers[i] for i in speaker_indices[:n_train])
val_speakers = set(speakers[i] for i in speaker_indices[n_train:n_train + n_val])
test_speakers = set(speakers[i] for i in speaker_indices[n_train + n_val:])

train_samples = [s for s in all_samples if s['speaker'] in train_speakers]
val_samples = [s for s in all_samples if s['speaker'] in val_speakers]
test_samples = [s for s in all_samples if s['speaker'] in test_speakers]

print(f"\nTrain: {len(train_samples)} samples from {len(train_speakers)} speakers")
print(f"Val:   {len(val_samples)} samples from {len(val_speakers)} speakers")
print(f"Test:  {len(test_samples)} samples from {len(test_speakers)} speakers")

for name, split in [("Train", train_samples), ("Val", val_samples), ("Test", test_samples)]:
    counts = Counter(s['label'] for s in split)
    print(f"\n{name}:")
    for i, cls in enumerate(CLASSES):
        print(f"  {cls}: {counts.get(i, 0)}")

## Cell 7: Load ECAPA-TDNN Encoder & Pre-Extract Embeddings

Extract 192-dim embeddings from the frozen ECAPA-TDNN encoder for all clips.
This takes ~10-15 minutes but makes Phase 1 training instant.

In [ ]:
import torch
import torchaudio
from speechbrain.inference.speaker import EncoderClassifier
from tqdm import tqdm
import soundfile as sf
import librosa

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ECAPA-TDNN
print("Loading ECAPA-TDNN encoder (first time downloads ~90MB)...")
encoder = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    run_opts={"device": str(device)},
)
print("Encoder loaded.")


def extract_embedding(filepath, encoder, target_sr=16000):
    """Extract 192-dim embedding from a single audio file."""
    try:
        audio, sr = sf.read(filepath)
        if len(audio.shape) > 1:
            audio = audio.mean(axis=1)
        audio = audio.astype(np.float32)

        # Resample if needed
        if sr != target_sr:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)

        # Must be at least 0.5 seconds
        if len(audio) < target_sr * 0.5:
            audio = np.pad(audio, (0, target_sr - len(audio)))

        waveform = torch.tensor(audio, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            embedding = encoder.encode_batch(waveform)
        return embedding.squeeze().cpu().numpy()  # (192,)
    except Exception as e:
        return None


# Extract embeddings for all samples
print(f"\nExtracting embeddings for {len(all_samples)} samples...")
print("This takes 10-15 minutes on T4.\n")

embeddings = {}
failed = 0
for sample in tqdm(all_samples, desc="Extracting"):
    emb = extract_embedding(sample['path'], encoder)
    if emb is not None:
        embeddings[sample['path']] = emb
    else:
        failed += 1

print(f"\nExtracted: {len(embeddings)} embeddings")
print(f"Failed: {failed}")
print(f"Embedding shape: {list(embeddings.values())[0].shape}")

# Filter samples to only those with valid embeddings
train_samples = [s for s in train_samples if s['path'] in embeddings]
val_samples = [s for s in val_samples if s['path'] in embeddings]
test_samples = [s for s in test_samples if s['path'] in embeddings]
print(f"\nAfter filtering: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")

## Cell 8: Dataset & DataLoader for Embedding-Based Training

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class EmbeddingDataset(Dataset):
    """Dataset of pre-extracted ECAPA-TDNN embeddings."""

    def __init__(self, samples, embeddings_dict):
        self.samples = samples
        self.embeddings = embeddings_dict

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        emb = torch.tensor(self.embeddings[s['path']], dtype=torch.float32)
        label = s['label']
        return emb, label


class AudioDataset(Dataset):
    """Dataset that loads raw audio (for Phase 2 end-to-end fine-tuning)."""

    def __init__(self, samples, target_sr=16000, max_len_sec=4.0, augment=False):
        self.samples = samples
        self.sr = target_sr
        self.max_len = int(max_len_sec * target_sr)
        self.augment = augment

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        try:
            audio, sr = sf.read(s['path'])
            if len(audio.shape) > 1:
                audio = audio.mean(axis=1)
            audio = audio.astype(np.float32)
            if sr != self.sr:
                audio = librosa.resample(audio, orig_sr=sr, target_sr=self.sr)
        except Exception:
            audio = np.zeros(self.max_len, dtype=np.float32)

        # Augmentation
        if self.augment:
            if np.random.random() < 0.3:
                snr_db = np.random.uniform(15, 30)
                sig_power = np.mean(audio ** 2)
                noise_power = sig_power / (10 ** (snr_db / 10))
                audio = audio + np.random.normal(0, np.sqrt(max(noise_power, 1e-10)), len(audio)).astype(np.float32)
            if np.random.random() < 0.3:
                factor = np.random.uniform(0.9, 1.1)
                indices = np.arange(0, len(audio), factor).astype(int)
                indices = indices[indices < len(audio)]
                audio = audio[indices]

        # Pad or truncate
        if len(audio) > self.max_len:
            start = np.random.randint(0, len(audio) - self.max_len) if self.augment else 0
            audio = audio[start:start + self.max_len]
        elif len(audio) < self.max_len:
            audio = np.pad(audio, (0, self.max_len - len(audio)))

        return torch.tensor(audio, dtype=torch.float32), s['label']


# Create embedding datasets for Phase 1
BATCH_SIZE = 64  # Embeddings are small, can use large batch

train_emb_ds = EmbeddingDataset(train_samples, embeddings)
val_emb_ds = EmbeddingDataset(val_samples, embeddings)
test_emb_ds = EmbeddingDataset(test_samples, embeddings)

train_emb_loader = DataLoader(train_emb_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_emb_loader = DataLoader(val_emb_ds, batch_size=BATCH_SIZE, shuffle=False)
test_emb_loader = DataLoader(test_emb_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Embedding datasets ready.")
print(f"Train batches: {len(train_emb_loader)}")
print(f"Val batches: {len(val_emb_loader)}")

# Verify
batch_emb, batch_labels = next(iter(train_emb_loader))
print(f"\nBatch embedding shape: {batch_emb.shape}")  # (64, 192)
print(f"Batch labels shape: {batch_labels.shape}")
print(f"Label example: {batch_labels[:5]}")

## Cell 9: Emotion Head Model

In [ ]:
import torch.nn as nn


class EmotionHead(nn.Module):
    """MLP head: 192-dim ECAPA embedding -> 5 emotion classes."""

    def __init__(self, input_dim=192, num_classes=5, dropout=0.3):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(x)


head = EmotionHead(input_dim=192, num_classes=NUM_CLASSES).to(device)
total_params = sum(p.numel() for p in head.parameters())
print(f"Emotion head: {total_params:,} parameters")

## Cell 10: Phase 1 — Train Head Only (Frozen Encoder)

Fast training on pre-extracted embeddings. Should reach UAR ~0.50-0.55 in minutes.

In [ ]:
from sklearn.metrics import recall_score
import time

# Class weights for imbalance
label_counts_train = Counter(s['label'] for s in train_samples)
total_train = len(train_samples)
weights = torch.tensor(
    [total_train / (NUM_CLASSES * label_counts_train.get(i, 1)) for i in range(NUM_CLASSES)],
    dtype=torch.float32
).to(device)
print("Class weights:")
for i, cls in enumerate(CLASSES):
    print(f"  {cls}: {weights[i]:.2f}")

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(head.parameters(), lr=1e-3, weight_decay=1e-2)

PHASE1_EPOCHS = 50
best_uar = 0.0
patience = 0
PATIENCE = 10


def compute_uar(model, loader, device):
    """Compute Unweighted Average Recall."""
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for emb, labels in loader:
            emb = emb.to(device)
            logits = model(emb)
            preds = logits.argmax(dim=-1).cpu()
            all_preds.extend(preds.tolist())
            all_true.extend(labels.tolist())
    return recall_score(all_true, all_preds, average='macro', zero_division=0)


print(f"\n{'='*60}")
print(f"PHASE 1: Head-only training ({PHASE1_EPOCHS} epochs max)")
print(f"{'='*60}\n")

for epoch in range(1, PHASE1_EPOCHS + 1):
    head.train()
    train_loss = 0.0

    for emb, labels in train_emb_loader:
        emb, labels = emb.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = head(emb)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_emb_loader)
    val_uar = compute_uar(head, val_emb_loader, device)

    if epoch % 5 == 0 or val_uar > best_uar:
        print(f"Epoch {epoch:3d} | Loss: {train_loss:.4f} | Val UAR: {val_uar:.4f}", end="")

    if val_uar > best_uar:
        best_uar = val_uar
        patience = 0
        torch.save({
            'head_state_dict': head.state_dict(),
            'epoch': epoch,
            'val_uar': best_uar,
            'classes': CLASSES,
            'phase': 1,
        }, f"{SAVE_DIR}/phase1_best.pt")
        if epoch % 5 == 0 or True:
            print(f" << BEST")
    else:
        patience += 1
        if epoch % 5 == 0:
            print(f" ({patience}/{PATIENCE})")
        if patience >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

print(f"\nPhase 1 complete. Best Val UAR: {best_uar:.4f}")

## Cell 11: Phase 2 — Fine-Tune Encoder + Head End-to-End

Unfreeze top TDNN layers and fine-tune with differential learning rate.
Takes 1-2 hours on T4. Should push UAR to 0.58+.

In [ ]:
import torch.nn as nn


class EmotionModelE2E(nn.Module):
    """End-to-end model: ECAPA-TDNN encoder + emotion head."""

    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, audio):
        # audio: (batch, samples)
        with torch.no_grad() if not self.training else torch.enable_grad():
            # SpeechBrain encoder expects (batch, samples)
            emb = self.encoder.encode_batch(audio.unsqueeze(1) if audio.dim() == 1 else audio)
            emb = emb.squeeze(1)  # (batch, 192)
        logits = self.head(emb)
        return logits


# Load best Phase 1 head
phase1_ckpt = torch.load(f"{SAVE_DIR}/phase1_best.pt", map_location=device, weights_only=True)
head.load_state_dict(phase1_ckpt['head_state_dict'])
print(f"Loaded Phase 1 head (UAR={phase1_ckpt['val_uar']:.4f})")

# Build end-to-end model
e2e_model = EmotionModelE2E(encoder, head).to(device)

# Freeze all encoder params first, then unfreeze top layers
for param in encoder.mods.parameters():
    param.requires_grad = False

# Unfreeze the last few TDNN blocks
# ECAPA-TDNN structure: conv layers -> TDNN blocks -> pooling
unfrozen = 0
for name, param in encoder.mods.named_parameters():
    # Unfreeze attention, last SE-Res2Net block, and pooling layers
    if any(k in name for k in ['layer4', 'layer5', 'attention', 'bn', 'fc', 'asp']):
        param.requires_grad = True
        unfrozen += 1

total_params = sum(p.numel() for p in e2e_model.parameters())
trainable = sum(p.numel() for p in e2e_model.parameters() if p.requires_grad)
print(f"\nTotal params: {total_params:,}")
print(f"Trainable: {trainable:,} ({100*trainable/total_params:.1f}%)")
print(f"Unfrozen encoder params: {unfrozen}")

# Create audio dataloaders for end-to-end training
BATCH_SIZE_E2E = 16

train_audio_ds = AudioDataset(train_samples, augment=True)
val_audio_ds = AudioDataset(val_samples, augment=False)

train_audio_loader = DataLoader(train_audio_ds, batch_size=BATCH_SIZE_E2E,
                                shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_audio_loader = DataLoader(val_audio_ds, batch_size=BATCH_SIZE_E2E,
                              shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain batches: {len(train_audio_loader)}")
print(f"Val batches: {len(val_audio_loader)}")

In [ ]:
from transformers import get_cosine_schedule_with_warmup

# Differential LR
encoder_params = [p for p in encoder.mods.parameters() if p.requires_grad]
head_params = list(head.parameters())

optimizer_e2e = torch.optim.AdamW([
    {"params": encoder_params, "lr": 1e-5, "weight_decay": 0.01},
    {"params": head_params, "lr": 5e-4, "weight_decay": 0.01},
])

PHASE2_EPOCHS = 30
total_steps = len(train_audio_loader) * PHASE2_EPOCHS
warmup_steps = int(0.1 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer_e2e,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

scaler = torch.amp.GradScaler()
best_uar_e2e = best_uar  # Start from Phase 1 best
patience = 0
PATIENCE_E2E = 7


def compute_uar_audio(model, loader, device):
    """Compute UAR from audio dataloader."""
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for audio, labels in loader:
            audio = audio.to(device)
            with torch.amp.autocast(device_type='cuda'):
                emb = encoder.encode_batch(audio)
                emb = emb.squeeze(1)
                logits = head(emb)
            preds = logits.argmax(dim=-1).cpu()
            all_preds.extend(preds.tolist())
            all_true.extend(labels.tolist())
    return recall_score(all_true, all_preds, average='macro', zero_division=0)


print(f"{'='*60}")
print(f"PHASE 2: End-to-end fine-tuning ({PHASE2_EPOCHS} epochs)")
print(f"{'='*60}\n")

for epoch in range(1, PHASE2_EPOCHS + 1):
    head.train()
    encoder.mods.train()
    train_losses = []
    epoch_start = time.time()

    for step, (audio, labels) in enumerate(train_audio_loader):
        audio, labels = audio.to(device), labels.to(device)
        optimizer_e2e.zero_grad()

        with torch.amp.autocast(device_type='cuda'):
            emb = encoder.encode_batch(audio)
            emb = emb.squeeze(1)
            logits = head(emb)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer_e2e)
        torch.nn.utils.clip_grad_norm_(list(encoder.mods.parameters()) + list(head.parameters()), max_norm=1.0)
        scaler.step(optimizer_e2e)
        scaler.update()
        scheduler.step()

        train_losses.append(loss.item())

        if (step + 1) % 50 == 0:
            print(f"  Epoch {epoch} Step {step+1}/{len(train_audio_loader)} "
                  f"Loss: {np.mean(train_losses[-50:]):.4f}")

    val_uar = compute_uar_audio(e2e_model, val_audio_loader, device)
    epoch_time = time.time() - epoch_start

    print(f"\nEpoch {epoch}/{PHASE2_EPOCHS} ({epoch_time:.0f}s)")
    print(f"  Train Loss: {np.mean(train_losses):.4f}")
    print(f"  Val UAR:    {val_uar:.4f}")

    if val_uar > best_uar_e2e:
        best_uar_e2e = val_uar
        patience = 0
        torch.save({
            'head_state_dict': head.state_dict(),
            'epoch': epoch,
            'val_uar': best_uar_e2e,
            'classes': CLASSES,
            'phase': 2,
        }, f"{SAVE_DIR}/best_model.pt")
        print(f"  >>> NEW BEST (UAR={best_uar_e2e:.4f})")
    else:
        patience += 1
        print(f"  No improvement ({patience}/{PATIENCE_E2E})")
        if patience >= PATIENCE_E2E:
            print(f"\nEarly stopping at epoch {epoch}")
            break

    print()

print(f"\nPhase 2 complete. Best Val UAR: {best_uar_e2e:.4f}")

## Cell 12: Test Set Evaluation

In [ ]:
from sklearn.metrics import recall_score, classification_report, confusion_matrix

# Load best model
best_ckpt = torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device, weights_only=True)
head.load_state_dict(best_ckpt['head_state_dict'])
head.eval()
print(f"Loaded best model from Phase {best_ckpt['phase']}, epoch {best_ckpt['epoch']}")

# Test on audio
test_audio_ds = AudioDataset(test_samples, augment=False)
test_audio_loader = DataLoader(test_audio_ds, batch_size=16, shuffle=False, num_workers=2)

all_preds, all_true = [], []
with torch.no_grad():
    for audio, labels in test_audio_loader:
        audio = audio.to(device)
        with torch.amp.autocast(device_type='cuda'):
            emb = encoder.encode_batch(audio)
            emb = emb.squeeze(1)
            logits = head(emb)
        preds = logits.argmax(dim=-1).cpu()
        all_preds.extend(preds.tolist())
        all_true.extend(labels.tolist())

test_uar = recall_score(all_true, all_preds, average='macro', zero_division=0)

print(f"\n{'='*60}")
print(f"FINAL TEST SET RESULTS")
print(f"{'='*60}")
print(f"Test UAR: {test_uar:.4f}")
print(f"Target:   >= 0.58")
print(f"Result:   {'PASSED' if test_uar >= 0.58 else 'BELOW TARGET'}")

print(f"\n{classification_report(all_true, all_preds, target_names=CLASSES, zero_division=0)}")

# Confusion matrix
cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:")
print(f"{'':15s}", end="")
for cls in CLASSES:
    print(f"{cls[:5]:>8s}", end="")
print()
for i, cls in enumerate(CLASSES):
    print(f"{cls:15s}", end="")
    for j in range(NUM_CLASSES):
        print(f"{cm[i][j]:8d}", end="")
    print()

if test_uar < 0.58:
    print("\nSuggestions:")
    print("  1. Unfreeze more encoder layers")
    print("  2. Increase Phase 2 epochs to 50")
    print("  3. Try lower encoder LR (5e-6)")
    print("  4. Add more augmentation (pitch shift, reverb)")

## Cell 13: Save to Google Drive

In [ ]:
import json

# Save test metrics
with open(f"{SAVE_DIR}/test_metrics.json", "w") as f:
    json.dump({
        "test_uar": test_uar,
        "classes": CLASSES,
        "num_classes": NUM_CLASSES,
        "phase": int(best_ckpt['phase']),
    }, f, indent=2)

print(f"Saved to {SAVE_DIR}/:")
!ls -lh {SAVE_DIR}/

print(f"\n{'='*60}")
print(f"DONE! Download best_model.pt from Google Drive to your Mac:")
print(f"  Place at: ~/Desktop/Claude-assistant/models/vocal_emotion/best_model.pt")
print(f"{'='*60}")